In [2]:
from pathlib import Path
from datetime import datetime

In [3]:
# ============================================================================
# PROJECT PATHS
# ============================================================================

# Base project directory
BASE_DIR = Path(r'C:\projects\aqi_pipline_2026-01-08')

# Input/Output directories
PATHS = {
    
    #download dir
    'download': BASE_DIR / '01_Downloaded',

    # Input data
    'input': BASE_DIR / '02_Input',
    'ancillary': BASE_DIR / '00_Ancillary_Data',
    'boundary_vector': BASE_DIR / '00_Ancillary_Data' / 'AbuDhabi_Regions.shp',

    # Ground station data
    'ground_station_csv': BASE_DIR / '00_Ancillary_Data' / 'AQ_HOURLY_202401_SCAD_DB_UG.csv',
    'old_ground_csv': BASE_DIR / '00_Ancillary_Data' / 'EAD_Hourly_2024_AQ_Points_AQI.csv',

    # Output directories
    'output_base': BASE_DIR / '03_Output_Data',
    'hourly_data': BASE_DIR / '03_Output_Data' / 'hourly_data',
    'daily_data': BASE_DIR / '03_Output_Data' / 'daily_data',
    'monthly_data': BASE_DIR / '03_Output_Data' / 'monthly_data',
    'quarterly_data': BASE_DIR / '03_Output_Data' / 'quarterly_data',
    'yearly_data': BASE_DIR / '03_Output_Data' / 'yearly_data',
    'aqi_data': BASE_DIR / '03_Output_Data' / 'aqi_data',
    'validation': BASE_DIR / '03_Output_Data' / 'validation',
    'validation_distribution': BASE_DIR / '03_Output_Data' / 'validation_distribution',
}

# Create output directories if they don't exist
for path_name, path_value in PATHS.items():
    if 'output' in path_name.lower() or path_name in ['hourly_data','daily_data', 'monthly_data', 'quarterly_data', 'yearly_data', 'aqi_data', 'validation', 'validation_distribution']:
        path_value.mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================================
# DATE RANGE SETTINGS
# ============================================================================

CONFIG = {
    # Date range for processing
    'start_date': '2025-12-01',  # Change this to your desired start date
    'end_date': '2025-12-31',    # Change this to your desired end date

    # Sampling period options: '1D' (daily), '1ME' (monthly), '1QE' (quarterly), '1YE' (yearly), '8H', '1H'
    'sample_period': '1ME',

    # Statistical method: 'mean', 'median', 'max'
    'stat_method': 'mean',

    # Region selection: 'All', 'Al Ain', 'Al Dhafra', 'Abu Dhabi'
    'region': 'All',

    # Region field name in shapefile
    'region_field': 'REGIONNAME',

    # Pollutants to process
    'pollutant_names': ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10'],

    # Target CRS
    'target_crs': 'EPSG:3857',
}

In [4]:
# ============================================================================
# AQI SETTINGS
# ============================================================================

AQI_CONFIG = {
    # AQI category definitions
    'categories': {
        'Good (0-50)': (0, 50),
        'Moderate (51-100)': (51, 100),
        'USG (101-150)': (101, 150),
        'Unhealthy (151-200)': (151, 200),
        'Very Unhealthy (201-300)': (201, 300),
        'Hazardous (301-500)': (301, 500)
    },

    # AQI color scheme (EPA standard)
    'colors': {
        'Good (0-50)': '#00E400',           # Green
        'Moderate (51-100)': '#FFFF00',     # Yellow
        'USG (101-150)': '#FF7E00',         # Orange
        'Unhealthy (151-200)': '#FF0000',   # Red
        'Very Unhealthy (201-300)': '#8F3F97',  # Purple
        'Hazardous (301-500)': '#7E0023'    # Maroon
    }
}

In [5]:
# ============================================================================
# PROCESSING SETTINGS
# ============================================================================

PROCESSING = {
    # Data normalization settings
    'filter_negative': True,  # Filter out negative/zero values
    'nodata_value': -9999,    # NoData value for TIFF exports

    # Chunk settings for Dask
    'chunks': 'auto',

    # Export settings
    'export_tiff_dtype': 'float32',
    'export_category_dtype': 'int16',
}

In [6]:
# ============================================================================
# STATION COORDINATES (for point-to-point validation)
# ============================================================================

STATION_COORDS = {
    # Station name: {'lat': latitude, 'lon': longitude}
    'Khadejah School': {'lat': 24.48156058, 'lon': 54.36929173},
    'Khalifa School': {'lat': 24.4815671, 'lon': 54.3692877},
    'Al Ain School': {'lat': 24.2190974, 'lon': 55.7348168},
    'Al Ain Street': {'lat': 24.2259397, 'lon': 55.7656666},
    'Al Mafraq': {'lat': 24.2862522, 'lon': 54.588901},
    'Al Maqta': {'lat': 24.4035015, 'lon': 54.5160884},
    'Al Qua\'a': {'lat': 23.5311228, 'lon': 55.4860383},
    'Al Tawia': {'lat': 24.2591303, 'lon': 55.7049288},
    'Baniyas': {'lat': 24.321383, 'lon': 54.635767},
    'Bida Zayed': {'lat': 23.65233, 'lon': 53.703777},
    'E11 Road': {'lat': 24.0352097, 'lon': 53.8851825},
    'Gayathi': {'lat': 23.8355039, 'lon': 52.8103723},
    'Habshan': {'lat': 23.7503774, 'lon': 53.7452404},
    'Hamdan Street': {'lat': 24.488983, 'lon': 54.3636678},
    'Khalifa City': {'lat': 24.4200138, 'lon': 54.5781332},
    'Liwa': {'lat': 23.095644, 'lon': 53.606229},
    'Mussafah': {'lat': 24.347317, 'lon': 54.502809},
    'Ruwais Transco': {'lat': 24.090855, 'lon': 52.754778},
    'Sweihan': {'lat': 24.466426, 'lon': 55.342792},
    'Zakher': {'lat': 24.1635902, 'lon': 55.7022387},
}

In [7]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_output_path(sample_period):
    """
    Get the appropriate output path based on sample period.

    Args:
        sample_period: '1D', '1ME', '1QE', '1YE', etc.

    Returns:
        Path object for output directory
    """
    period_map = {
        '1D': PATHS['daily_data'],
        '1ME': PATHS['monthly_data'],
        '1QE': PATHS['quarterly_data'],
        '1YE': PATHS['yearly_data'],
        '8H': PATHS['output_base'] / 'hourly_8h_data',
        '1H': PATHS['output_base'] / 'hourly_1h_data',
    }

    output_path = period_map.get(sample_period, PATHS['output_base'] / f'{sample_period}_data')
    output_path.mkdir(parents=True, exist_ok=True)

    return output_path


def print_config():
    """Print current configuration settings."""
    print("=" * 70)
    print("CURRENT CONFIGURATION")
    print("=" * 70)
    print(f"\nDate Range:")
    print(f"  Start: {CONFIG['start_date']}")
    print(f"  End: {CONFIG['end_date']}")
    print(f"\nProcessing Settings:")
    print(f"  Sample Period: {CONFIG['sample_period']}")
    print(f"  Statistical Method: {CONFIG['stat_method']}")
    print(f"  Region: {CONFIG['region']}")
    print(f"  Pollutants: {', '.join(CONFIG['pollutant_names'])}")
    print(f"\nOutput Path:")
    print(f"  {get_output_path(CONFIG['sample_period'])}")
    print("=" * 70)


def update_config(start_date=None, end_date=None, sample_period=None,
                  stat_method=None, region=None, pollutants=None):
    """
    Update configuration settings programmatically.

    Args:
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        sample_period: '1D', '1ME', '1QE', '1YE', etc.
        stat_method: 'mean', 'median', 'max'
        region: 'All', 'Al Ain', 'Al Dhafra', 'Abu Dhabi'
        pollutants: List of pollutant names
    """
    if start_date:
        CONFIG['start_date'] = start_date
    if end_date:
        CONFIG['end_date'] = end_date
    if sample_period:
        CONFIG['sample_period'] = sample_period
    if stat_method:
        CONFIG['stat_method'] = stat_method
    if region:
        CONFIG['region'] = region
    if pollutants:
        CONFIG['pollutant_names'] = pollutants

    print("Configuration updated!")
    print_config()

In [8]:
# ============================================================================
# AUTO-PRINT CONFIG ON LOAD
# ============================================================================

if __name__ != '__main__':
    print("\n✓ Configuration loaded successfully!")
    print(f"  Base directory: {BASE_DIR}")
    print(f"  Date range: {CONFIG['start_date']} to {CONFIG['end_date']}")
    print(f"  Sample period: {CONFIG['sample_period']}")
    print(f"\n  Use print_config() to see all settings")
    print(f"  Use update_config() to change settings\n")